In [ ]:
import torch
import numpy as np
import cv2
from facenet_pytorch import MTCNN

print("✅ PyTorch Version:", torch.__version__)
print("✅ CUDA Available:", torch.cuda.is_available())
print("✅ GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found")
print("✅ NumPy Version:", np.__version__)
print("✅ OpenCV Version:", cv2.__version__)
print("✅ Facenet-PyTorch Loaded:", MTCNN is not None)

c:\Users\bchal\anaconda3\envs\mindcare-ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ PyTorch Version: 2.6.0+cu126
✅ CUDA Available: True
✅ GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
✅ NumPy Version: 1.26.4
✅ OpenCV Version: 4.11.0
✅ Facenet-PyTorch Loaded: True


In [4]:
# vital_signs_api.py
from flask import Flask, request, jsonify
import numpy as np
import cv2
import base64
import time
from scipy.signal import butter, filtfilt
import os
import logging
import requests
import json

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = Flask(__name__)

# Express.js auth server details
AUTH_SERVER = "http://your-express-server:3000"
MONGODB_URI = "mongodb://127.0.0.1:27017/mindcare"

# Initialize face detector
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# MongoDB connection
try:
    from pymongo import MongoClient
    client = MongoClient(MONGODB_URI)
    db = client.get_database()
    vital_signs_collection = db.vitalSigns
    logger.info("MongoDB connection established")
except Exception as e:
    logger.error(f"MongoDB connection error: {str(e)}")
    vital_signs_collection = None

def verify_token(token):
    """Verify JWT token with Express.js auth server"""
    try:
        response = requests.post(
            f"{AUTH_SERVER}/api/verify-token",
            json={"token": token},
            headers={"Content-Type": "application/json"}
        )
        if response.status_code == 200:
            return response.json()
        return None
    except Exception as e:
        logger.error(f"Token verification error: {str(e)}")
        return None

def preprocess_image(base64_image):
    """Convert base64 image to numpy array"""
    try:
        # Check if the base64 string contains the header
        if ',' in base64_image:
            base64_image = base64_image.split(',')[1]
            
        img_data = base64.b64decode(base64_image)
        nparr = np.frombuffer(img_data, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        return img
    except Exception as e:
        logger.error(f"Error preprocessing image: {str(e)}")
        return None

def extract_face_roi(image):
    """Extract face region of interest"""
    if image is None:
        return None
        
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # Use OpenCV for face detection
    try:
        faces = face_cascade.detectMultiScale(gray, 1.1, 4)
        
        if len(faces) == 0:
            return None
        
        # Use the first detected face
        x, y, w, h = faces[0]
        face_roi = image[y:y+h, x:x+w]
        
        return face_roi, (x, y, w, h)
    except Exception as e:
        logger.error(f"Face detection error: {str(e)}")
        return None

def calculate_heart_rate(face_roi_sequence, time_sequence, fps=30):
    """Calculate heart rate from a sequence of face ROIs"""
    try:
        if len(face_roi_sequence) < fps * 3:  # Need at least 3 seconds of data
            return 68  # Default value if not enough data
        
        # Extract green channel and calculate mean
        green_vals = []
        for frame in face_roi_sequence:
            if frame is not None and frame.size > 0:
                green_vals.append(np.mean(frame[:, :, 1]))
        
        if len(green_vals) < fps * 3:
            return 68
        
        # Apply bandpass filter (0.7-3.0 Hz for heart rate 42-180 BPM)
        nyq = 0.5 * fps
        low = 0.7 / nyq
        high = 3.0 / nyq
        b, a = butter(2, [low, high], btype='band')
        green_filtered = filtfilt(b, a, green_vals)
        
        # Find peaks
        from scipy.signal import find_peaks
        peaks, _ = find_peaks(green_filtered, distance=fps/2)
        
        if len(peaks) < 2:
            return 68  # Default fallback
        
        # Calculate average time between peaks
        peak_times = [time_sequence[p] for p in peaks]
        intervals = np.diff(peak_times)
        avg_interval = np.mean(intervals)
        
        # Convert to BPM
        heart_rate = 60.0 / avg_interval if avg_interval > 0 else 68
        
        # Reasonability check
        if heart_rate < 40 or heart_rate > 200:
            return 68
        
        return int(heart_rate)
    except Exception as e:
        logger.error(f"Error calculating heart rate: {str(e)}")
        return 68

def estimate_bp(heart_rate):
    """Estimate blood pressure based on heart rate"""
    try:
        sys_bp = int(90 + (heart_rate * 0.6))
        dia_bp = int(60 + (heart_rate * 0.25))
        
        return sys_bp, dia_bp
    except Exception as e:
        logger.error(f"Error estimating blood pressure: {str(e)}")
        return 134, 85

# Store recent frames for processing
frame_buffers = {}  # Dict to store buffers for each user
time_buffers = {}
MAX_BUFFER_SIZE = 150  # 5 seconds at 30fps

def save_vital_signs(user_id, heart_rate, systolic_bp, diastolic_bp):
    """Save vital signs data to MongoDB"""
    if vital_signs_collection is None:
        return False
        
    try:
        record = {
            "userId": user_id,
            "timestamp": time.time(),
            "date": time.strftime("%Y-%m-%d %H:%M:%S"),
            "heartRate": heart_rate,
            "systolicBP": systolic_bp,
            "diastolicBP": diastolic_bp
        }
        vital_signs_collection.insert_one(record)
        return True
    except Exception as e:
        logger.error(f"Error saving to MongoDB: {str(e)}")
        return False

@app.route('/api/analyze', methods=['POST'])
def analyze_vital_signs():
    try:
        if not request.is_json:
            return jsonify({'error': 'Request must be JSON'}), 400
            
        data = request.get_json()
        
        # Extract and verify token
        auth_header = request.headers.get('Authorization')
        if not auth_header or not auth_header.startswith('Bearer '):
            return jsonify({'error': 'Authorization token required'}), 401
            
        token = auth_header.split(' ')[1]
        user_data = verify_token(token)
        
        if not user_data:
            return jsonify({'error': 'Invalid or expired token'}), 401
            
        user_id = user_data.get('userId')
        
        if 'image' not in data:
            return jsonify({'error': 'No image provided'}), 400
        
        base64_image = data['image']
        image = preprocess_image(base64_image)
        
        if image is None:
            return jsonify({'error': 'Invalid image data'}), 400
        
        # Process face
        face_data = extract_face_roi(image)
        if face_data is None:
            return jsonify({
                'error': 'No face detected', 
                'heart_rate': 68, 
                'systolic_bp': 134, 
                'diastolic_bp': 85
            }), 200
        
        face_roi, face_coords = face_data
        
        # Initialize buffers for this user if they don't exist
        if user_id not in frame_buffers:
            frame_buffers[user_id] = []
            time_buffers[user_id] = []
        
        # Add to user's buffer
        current_time = time.time()
        frame_buffers[user_id].append(face_roi)
        time_buffers[user_id].append(current_time)
        
        # Maintain buffer size
        if len(frame_buffers[user_id]) > MAX_BUFFER_SIZE:
            frame_buffers[user_id].pop(0)
            time_buffers[user_id].pop(0)
        
        # Calculate vital signs if we have enough frames
        if len(frame_buffers[user_id]) >= 90:  # At least 3 seconds of data
            heart_rate = calculate_heart_rate(frame_buffers[user_id], time_buffers[user_id])
            sys_bp, dia_bp = estimate_bp(heart_rate)
        else:
            # Not enough data yet, return placeholders
            heart_rate = 68
            sys_bp, dia_bp = 134, 85
        
        # Save data to MongoDB if save flag is set
        should_save = data.get('saveData', False)
        if should_save:
            save_result = save_vital_signs(user_id, heart_rate, sys_bp, dia_bp)
        else:
            save_result = False
        
        response = {
            'heart_rate': heart_rate,
            'systolic_bp': sys_bp,
            'diastolic_bp': dia_bp,
            'face_coordinates': {
                'x': face_coords[0],
                'y': face_coords[1],
                'width': face_coords[2],
                'height': face_coords[3]
            },
            'dataSaved': save_result
        }
        
        return jsonify(response)
    except Exception as e:
        logger.error(f"Error in analyze_vital_signs: {str(e)}")
        return jsonify({
            'error': 'Internal server error',
            'heart_rate': 68,
            'systolic_bp': 134,
            'diastolic_bp': 85
        }), 500

@app.route('/health', methods=['GET'])
def health_check():
    """Simple health check endpoint"""
    return jsonify({'status': 'ok'})

@app.route('/', methods=['GET'])
def index():
    """Root endpoint with API info"""
    return jsonify({
        'message': 'Vital Signs API Server',
        'endpoints': {
            'health_check': '/health (GET)',
            'analyze': '/api/analyze (POST)'
        }
    })

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=False)

INFO:__main__:MongoDB connection established


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://10.6.68.72:5000
INFO:werkzeug:Press CTRL+C to quit
